In [ ]:
from google.colab import drive
import os
os.chdir('/content')
drive.mount('/content/drive', force_remount=True)
os.chdir('/content/drive/MyDrive/3dgs_decomposition')

In [ ]:
!pip install lpips

In [ ]:
import os
import sys
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import numpy as np
import csv
import pandas as pd
import gc
import torch

In [ ]:
!pip install -r intr_decomp_3dgs/requirements.txt
!pip install intr_decomp_3dgs/diff-gaussian-rasterization  --no-build-isolation -v

In [ ]:
!pip install plyfile

In [ ]:
!rm -rf /tmp/simple-knn
!git clone https://gitlab.inria.fr/bkerbl/simple-knn.git /tmp/simple-knn
!sed -i '1i #include <float.h>' /tmp/simple-knn/spatial.cu
!pip install /tmp/simple-knn --no-build-isolation

In [ ]:
import eval.colmap_reader as colmap_reader
import eval.loaders as loaders
eval_dir = os.path.dirname(os.path.abspath(loaders.__file__))
if eval_dir not in sys.path:
    sys.path.insert(0, eval_dir)
import eval.mv_metrics as mv_metrics
import eval.reprojection as reprojection
import eval.properties_3dgs as properties_3dgs

In [ ]:
predictors = ["rgbx", "marigold", "colourful_iid"]

In [ ]:
scenes = {"hypersim": ["ai_001_001", "ai_002_001"], "mipnerf360": ["bicycle", "garden", "room"]}

In [ ]:
root = "/content/drive/MyDrive/3dgs_decomposition"

In [ ]:
# save the depth map
for predictor in predictors:
    for dataset, scene_list in scenes.items():
        for scene in scene_list:
            model_dir = os.path.join(root, "baseline_train", predictor, dataset, scene)
            ply_path = os.path.join(model_dir, "point_cloud", "iteration_10000", "albedo.ply")
            out_depth = os.path.join(root, "eval", "depth_cache", predictor, dataset, scene)

            if not os.path.exists(ply_path):
                print(f"missing ply: {ply_path}")
                continue

            has_depth = os.path.isdir(out_depth) and any(file.endswith(".npy") for file in os.listdir(out_depth))
            if not has_depth:
                if dataset == "hypersim":
                    colmap_dir = os.path.join(root, "data", "hypersim", "colmap_format", scene, "cam_00")
                else:
                    colmap_dir = os.path.join(root, "data", "mipnerf360", scene)
                properties_3dgs.export_3dgs_depth(colmap_dir, model_dir, out_depth)
                print(f"done {predictor} {dataset}/{scene}")

In [ ]:
rows = []
scene_info = {}

In [ ]:
metrics = ["rmse", "mae", "psnr", "lpips"]

In [ ]:
import importlib
importlib.reload(reprojection)

In [ ]:
for predictor in predictors:
    for dataset, scene_list in scenes.items():
        for scene in scene_list:
            if (dataset, scene, predictor) not in scene_info:
                if dataset == "hypersim":
                    colmap_dir = os.path.join(root, "data", "hypersim", "colmap_format", scene, "cam_00")
                else:
                    colmap_dir = os.path.join(root, "data", "mipnerf360", scene)
                cams = colmap_reader.load_scene_cameras(colmap_dir)
                pairs_by_diff = reprojection.covisible_pairs(root, dataset, scene, cams, predictor)
                scene_info[(dataset, scene, predictor)] = (cams, pairs_by_diff)
            else:
                cams, pairs_by_diff = scene_info[(dataset, scene, predictor)]

            all_pairs = [(diff, i, j) for diff, pairs in pairs_by_diff.items() for i, j in pairs]
            if not all_pairs:
                sample_al = loaders.get_albedo_path(root, predictor, dataset, scene, cams[0])
                sample_de = loaders.get_depth_path(root, dataset, scene, cams[0], predictor)
                print(
                    f"no {predictor} on {dataset}/{scene} | "
                    f"albedo_exists={os.path.exists(sample_al)} ({sample_al}) | "
                    f"depth_exists={os.path.exists(sample_de)} ({sample_de})"
                )
                continue

            albedos = {}
            depths = {}
            thresh = 0.12
            prev_i = None

            for diff, i, j in sorted(all_pairs, key=lambda x: (x[1], x[0])):
                if prev_i is not None and i != prev_i:
                    albedos.pop(prev_i, None)
                    depths.pop(prev_i, None)
                prev_i = i

                for idx in (i, j):
                    if idx not in albedos:
                        al_path = loaders.get_albedo_path(root, predictor, dataset, scene, cams[idx])
                        de_path = loaders.get_depth_path(root, dataset, scene, cams[idx], predictor)
                        if os.path.exists(al_path) and os.path.exists(de_path):
                            albedos[idx] = loaders.load_albedo(al_path)
                            tgt_h, tgt_w = albedos[idx].shape[:2]
                            depths[idx] = loaders.load_depth(de_path, (tgt_h, tgt_w))

                if i in albedos and j in albedos:
                    warped_j, mask = reprojection.reproject_view(cams[i], cams[j], depths[i], depths[j], albedos[j], occ_thresh=0.15, edge_thresh=0.30)
                    metrics = mv_metrics.compute_mv_metrics(albedos[i], warped_j, mask)
                    if metrics["mod_m"] > 0:
                        rows.append({"predictor": predictor, "dataset": dataset, "scene": scene, "delta": diff, "src_idx": i, "tgt_idx": j, **metrics})

            del albedos, depths
            gc.collect()
            torch.cuda.empty_cache()
            print(f"{predictor} {dataset}/{scene} done")


In [ ]:
metrics_key = [("rmse", "aiming to minimise"), ("mae", "aiming to minimise"), ("psnr", "aiming to maximise"), ("lpips", "aiming to minimise")]

In [ ]:
def plot_scene_boxplots(delta: int = 8) -> Figure:
    sub = [r for r in rows if r["delta"] == delta]
    scene_pairs = [(ds, s) for ds, s_list in scenes.items() for s in s_list if any(r["dataset"] == ds and r["scene"] == s for r in sub)]
    if not scene_pairs:
        return

    n_rows, n_cols = len(metrics_key), len(scene_pairs)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3.0 * n_rows), squeeze=False)
    for i, (metric, metric_label) in enumerate(metrics_key):
        for j, (ds, scene) in enumerate(scene_pairs):
            ax = axes[i, j]
            box_data = [[r[metric] for r in sub if r["dataset"] == ds and r["scene"] == scene and r["predictor"] == pred] for pred in predictors]

            ax.boxplot(box_data, tick_labels=predictors if i == n_rows - 1 else [""] * len(predictors), patch_artist=True, boxprops={"facecolor": "papayawhip"})
            if i == n_rows - 1:
                ax.tick_params(axis="x", rotation=20)
            if i == 0:
                ax.set_title(f"{ds}\n{scene}")
            if j == 0:
                ax.set_ylabel(f"{metric.upper()} ({metric_label})")

    fig.suptitle(f"Multi-View Albedo Consistency Across Predictors (Frame offset = {delta})")
    plt.tight_layout()
    return fig


plot_scene_boxplots(delta=8);
plot_scene_boxplots(delta=2);


In [ ]:
os.makedirs(eval_dir, exist_ok=True)

csv_path = os.path.join(eval_dir, "mv_metrics.csv")
fieldnames = ["predictor", "dataset", "scene", "delta", "src_idx", "tgt_idx", "overlap", "mod_m", "rmse", "mae", "psnr", "lpips"]

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"saved {len(rows)} rows to {csv_path}")

fig8 = plot_scene_boxplots(delta=8)
plot_path8 = os.path.join(eval_dir, "mv_boxplots_delta8.png")
fig8.savefig(plot_path8, dpi=300, bbox_inches="tight")
print(f"saved plot to {plot_path8}")

fig2 = plot_scene_boxplots(delta=8)
plot_path2 = os.path.join(eval_dir, "mv_boxplots_delta2.png")
fig2.savefig(plot_path2, dpi=300, bbox_inches="tight")
print(f"saved plot to {plot_path2}")

In [ ]:
df = pd.DataFrame(rows)
raw_cols = ["rmse", "mae", "psnr", "lpips"]
metric_cols = ["rmse (↓)", "mae (↓)", "psnr (↑)", "lpips (↓)"]

df = pd.DataFrame(rows).rename(columns=dict(zip(raw_cols, metric_cols)))

summary = (df.groupby(["delta", "predictor"], sort=False)[metric_cols].mean().reindex(predictors, level="predictor").round(4))

print(summary.to_markdown())

summary_path = os.path.join(root, "eval", "summary_means.csv")
summary.to_csv(summary_path)
print(f"\nsaved summary table to {summary_path}")

In [ ]:
def plot_warped_comparison(dataset: str, scene: str, delta: int = 8, pair_idx: int = 0):
    ref = predictors[0]
    if (dataset, scene, ref) not in scene_info:
        if dataset == "hypersim":
            colmap_dir = os.path.join(root, "data", "hypersim", "colmap_format", scene, "cam_00")
        else:
            colmap_dir = os.path.join(root, "data", "mipnerf360", scene)

        cams = colmap_reader.load_scene_cameras(colmap_dir)
        scene_info[(dataset, scene, ref)] = (cams, reprojection.covisible_pairs(root, dataset, scene, cams, ref))

    cams, pairs_by_diff = scene_info[(dataset, scene, ref)]
    i, j = pairs_by_diff[delta][pair_idx]

    thresh = 0.12
    err_cmap = plt.get_cmap("inferno").copy()
    err_cmap.set_bad(color="#d9d9d9")

    fig, axes = plt.subplots(len(predictors), 4, figsize=(16.5, 3.8 * len(predictors)), squeeze=False, constrained_layout=True)

    for row_idx, pred in enumerate(predictors):
        a_i = loaders.load_albedo(loaders.get_albedo_path(root, pred, dataset, scene, cams[i]))
        a_j = loaders.load_albedo(loaders.get_albedo_path(root, pred, dataset, scene, cams[j]))
        d_i = loaders.load_depth(loaders.get_depth_path(root, dataset, scene, cams[i], pred), a_i.shape[:2])
        d_j = loaders.load_depth(loaders.get_depth_path(root, dataset, scene, cams[j], pred), a_j.shape[:2])

        warped_j, mask = reprojection.reproject_view(cams[i], cams[j], d_i, d_j, a_j, occ_thresh=0.15, edge_thresh=0.30)

        m = mv_metrics.compute_mv_metrics(a_i, warped_j, mask)

        warped_vis = np.where(mask[..., None], warped_j, 0.85)
        err_map = np.where(mask, np.mean(np.abs(a_i - warped_j), axis=-1), np.nan)

        axes[row_idx, 0].imshow(a_i)
        axes[row_idx, 0].set_ylabel(pred)
        axes[row_idx, 1].imshow(a_j)
        axes[row_idx, 2].imshow(warped_vis)
        im = axes[row_idx, 3].imshow(err_map, cmap=err_cmap, vmin=0.0, vmax=0.3)

        if row_idx == 0:
            axes[row_idx, 0].set_title(f"Source Albedo (view {i})", fontsize=10)
            axes[row_idx, 1].set_title(f"Target Albedo (view {j})", fontsize=10)

        header_w = "Warped Target → Source\n" if row_idx == 0 else ""
        axes[row_idx, 2].set_title(f"{header_w}(valid overlap: {m['overlap']:.1%})", fontsize=10)

        header = "Per-Pixel Abs Error (gray = occluded)\n" if row_idx == 0 else ""
        axes[row_idx, 3].set_title(f"{header}RMSE: {m['rmse']:.3f} | MAE: {m['mae']:.3f} | PSNR: {m['psnr']:.1f} | LPIPS: {m['lpips']:.3f}", fontsize=10)

        fig.colorbar(im, ax=axes[row_idx, 3], label="Mean |Δ Albedo| (sRGB)")
        for ax in axes[row_idx]:
            ax.set(xticks=[], yticks=[])

    fig.suptitle(f"Multi-View Warped Albedo Comparison — {dataset}/{scene} (Δ={delta}, pair {i}→{j})")
    out_dir = os.path.join(root, "eval")
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"warped_compare_{dataset}_{scene}_d{delta}_pair{pair_idx}.png")
    fig.savefig(out_path, dpi=250, bbox_inches="tight")
    print(f"saved to {out_path}")
    return fig


plot_warped_comparison("hypersim", "ai_001_001", delta=8, pair_idx=0);
plot_warped_comparison("mipnerf360", "bicycle", delta=2, pair_idx=0);

For comparison, now we warp the hypersim scenes based on their original ground-truth depth map.

In [ ]:
rows_gt = []

In [ ]:
scene_info_gt = {}

In [ ]:
thresh_gt = 0.05

In [ ]:
for predictor in predictors:
    for scene in scenes["hypersim"]:
        dataset = "hypersim"
        if (dataset, scene, predictor) not in scene_info_gt:
            colmap_dir = os.path.join(root, "data", "hypersim", "colmap_format", scene, "cam_00")
            cams = colmap_reader.load_scene_cameras(colmap_dir)
            pairs_by_diff = reprojection.covisible_pairs(root, dataset, scene, cams, predictor, use_gt_depth=True, thresh=thresh_gt)
            scene_info_gt[(dataset, scene, predictor)] = (cams, pairs_by_diff)
        else:
            cams, pairs_by_diff = scene_info_gt[(dataset, scene, predictor)]

        albedos = {}
        depths = {}

        for i, cam in enumerate(cams):
            al_path = loaders.get_albedo_path(root, predictor, dataset, scene, cam)
            de_path = loaders.get_gt_depth_path(root, dataset, scene, cam, predictor)
            if os.path.exists(al_path) and os.path.exists(de_path):
                albedos[i] = loaders.load_albedo(al_path)
                tgt_h, tgt_w = albedos[i].shape[:2]
                depths[i] = loaders.load_depth(de_path, (tgt_h, tgt_w))

        if not albedos:
            sample_al = loaders.get_albedo_path(root, predictor, dataset, scene, cams[0])
            sample_de = loaders.get_gt_depth_path(root, dataset, scene, cams[0], predictor)
            print(
                f"no {predictor} on {dataset}/{scene} | "
                f"albedo_exists={os.path.exists(sample_al)} ({sample_al}) | "
                f"depth_exists={os.path.exists(sample_de)} ({sample_de})"
            )
            continue

        for diff, pairs in pairs_by_diff.items():
            for i, j in pairs:
                if i in albedos and j in albedos:
                    warped_j, mask = reprojection.reproject_view(
                        cams[i], cams[j], depths[i], depths[j], albedos[j],
                        occ_thresh=thresh_gt, edge_thresh=thresh_gt,
                    )
                    metrics = mv_metrics.compute_mv_metrics(albedos[i], warped_j, mask)
                    if metrics["mod_m"] > 0:
                        rows_gt.append({"predictor": predictor, "dataset": dataset, "scene": scene, "delta": diff, "src_idx": i, "tgt_idx": j, **metrics})
        print(f"{predictor} {dataset}/{scene} (GT depth) done")

In [ ]:
df_gt = pd.DataFrame(rows_gt)
df_gt.to_csv(os.path.join(root, "eval", "mv_consistency_gt_hypersim.csv"), index=False)

In [ ]:
summary_gt = df_gt.groupby(["predictor", "delta"])[["overlap", "rmse", "mae", "psnr", "lpips"]].mean().round(4)
print(summary.to_markdown())

In [ ]:
summary_path_gt = os.path.join(root, "eval", "summary_means_gt.csv")
summary.to_csv(summary_path_gt)
print(f"\nsaved summary table to {summary_path_gt}")

In [ ]:
def plot_scene_boxplots_gt(delta: int = 8) -> Figure:
    sub = [r for r in rows_gt if r["delta"] == delta]
    scene_pairs = [(ds, s) for ds, s_list in scenes.items() for s in s_list if any(r["dataset"] == ds and r["scene"] == s for r in sub)]
    if not scene_pairs:
        return

    n_rows, n_cols = len(metrics_key), len(scene_pairs)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3.0 * n_rows), squeeze=False)
    for i, (metric, metric_label) in enumerate(metrics_key):
        for j, (ds, scene) in enumerate(scene_pairs):
            ax = axes[i, j]
            box_data = [[r[metric] for r in sub if r["dataset"] == ds and r["scene"] == scene and r["predictor"] == pred] for pred in predictors]

            ax.boxplot(box_data, tick_labels=predictors if i == n_rows - 1 else [""] * len(predictors), patch_artist=True, boxprops={"facecolor": "papayawhip"})
            if i == n_rows - 1:
                ax.tick_params(axis="x", rotation=20)
            if i == 0:
                ax.set_title(f"{ds}\n{scene}")
            if j == 0:
                ax.set_ylabel(f"{metric.upper()} ({metric_label})")

    fig.suptitle(f"Multi-View Albedo Consistency Across Predictors (Frame offset = {delta})")
    plt.tight_layout()
    return fig


plot_scene_boxplots_gt(delta=8);
plot_scene_boxplots_gt(delta=2);


In [ ]:
fig_gt = plot_scene_boxplots_gt(delta=8)
plot_path_gt = os.path.join(eval_dir, "mv_boxplots_gt.png")
fig_gt.savefig(plot_path_gt, dpi=300, bbox_inches="tight")
print(f"saved plot to {plot_path_gt}")

In [ ]:
def plot_warped_comparison_gt(dataset: str, scene: str, delta: int = 8, pair_idx: int = 0):
    ref = predictors[0]
    thresh = 0.05
    if (dataset, scene, ref) not in scene_info_gt:
        colmap_dir = os.path.join(root, "data", "hypersim", "colmap_format", scene, "cam_00")
        cams = colmap_reader.load_scene_cameras(colmap_dir)
        scene_info_gt[(dataset, scene, ref)] = (cams, reprojection.covisible_pairs(root, dataset, scene, cams, ref, use_gt_depth=True, thresh=thresh))

    cams, pairs_by_diff = scene_info_gt[(dataset, scene, ref)]
    i, j = pairs_by_diff[delta][pair_idx]

    err_cmap = plt.get_cmap("inferno").copy()
    err_cmap.set_bad(color="#d9d9d9")

    fig, axes = plt.subplots(len(predictors), 4, figsize=(16.5, 3.8 * len(predictors)), squeeze=False, constrained_layout=True)

    for row_idx, pred in enumerate(predictors):
        a_i = loaders.load_albedo(loaders.get_albedo_path(root, pred, dataset, scene, cams[i]))
        a_j = loaders.load_albedo(loaders.get_albedo_path(root, pred, dataset, scene, cams[j]))
        d_i = loaders.load_depth(loaders.get_gt_depth_path(root, dataset, scene, cams[i], pred), a_i.shape[:2])
        d_j = loaders.load_depth(loaders.get_gt_depth_path(root, dataset, scene, cams[j], pred), a_j.shape[:2])

        warped_j, mask = reprojection.reproject_view(cams[i], cams[j], d_i, d_j, a_j, occ_thresh=thresh, edge_thresh=thresh)
        m = mv_metrics.compute_mv_metrics(a_i, warped_j, mask)

        warped_vis = np.where(mask[..., None], warped_j, 0.85)
        err_map = np.where(mask, np.mean(np.abs(a_i - warped_j), axis=-1), np.nan)

        axes[row_idx, 0].imshow(a_i)
        axes[row_idx, 0].set_ylabel(pred)
        axes[row_idx, 1].imshow(a_j)
        axes[row_idx, 2].imshow(warped_vis)
        im = axes[row_idx, 3].imshow(err_map, cmap=err_cmap, vmin=0.0, vmax=0.3)

        if row_idx == 0:
            axes[row_idx, 0].set_title(f"Source Albedo (view {i})", fontsize=10)
            axes[row_idx, 1].set_title(f"Target Albedo (view {j})", fontsize=10)

        header_w = "Warped Target → Source\n" if row_idx == 0 else ""
        axes[row_idx, 2].set_title(f"{header_w}(valid overlap: {m['overlap']:.1%})", fontsize=10)

        header = "Per-Pixel Abs Error (gray = occluded)\n" if row_idx == 0 else ""
        axes[row_idx, 3].set_title(f"{header}RMSE: {m['rmse']:.3f} | MAE: {m['mae']:.3f} | PSNR: {m['psnr']:.1f} | LPIPS: {m['lpips']:.3f}", fontsize=10)

        fig.colorbar(im, ax=axes[row_idx, 3], label="Mean |Δ Albedo| (sRGB)")
        for ax in axes[row_idx]:
            ax.set(xticks=[], yticks=[])

    fig.suptitle(f"Multi-View Warped Albedo Comparison (GT Depth) — {dataset}/{scene} (Δ={delta}, pair {i}→{j})")
    out_dir = os.path.join(root, "eval")
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"warped_compare_gt_{dataset}_{scene}_d{delta}_pair{pair_idx}.png")
    fig.savefig(out_path, dpi=250, bbox_inches="tight")
    print(f"saved to {out_path}")
    return fig


for scene in scenes["hypersim"]:
    plot_warped_comparison_gt("hypersim", scene, delta=8, pair_idx=0)

Check how many pairs are used for these bits of data (based on pairs with right offset when they have more than 10% overlap)

In [ ]:
for (ds, s, pred), (cams, pairs_by_diff) in scene_info.items():
    print(f"{pred} | {ds}/{s} (N={len(cams)}): d=2 -> {len(pairs_by_diff[2])}, d=8 -> {len(pairs_by_diff[8])}")

In [ ]:
for (ds, s, pred), (cams, pairs_by_diff) in scene_info_gt.items():
    print(f"{pred} | {ds}/{s} (N={len(cams)}): d=2 -> {len(pairs_by_diff[2])}, d=8 -> {len(pairs_by_diff[8])}")